# ISIC 2019: recovered CNN baseline and validation

**Author:** Amir Albana  
**Study stage:** Historical eight-class experiments (AK, BCC, BKL, DF, MEL, NV, SCC, VASC). The current thesis uses a separate benign-versus-malignant task; these results must not be presented as binary results.

**Requirements:** Python with NumPy, pandas, scikit-learn, Pillow and TensorFlow. In Kaggle, attach Skin Cancer ISIC, ISIC 2019 Training Ground Truth and ISIC 2019 Metadata. Enable a GPU for training. Local source CSVs are used; Internet access is not required.

**Reading and execution:** Code cells are ordered for top-to-bottom use with those inputs attached. Outputs below are retained historical results, not a new run of this cleaned notebook. Execution counters were cleared during editing. Device logs, obsolete troubleshooting cells and duplicate setup attempts were removed; training logs were condensed without changing recorded metrics. Repeated audit examples were deduplicated for display. Original versions remain available in Git history. Rerunning training can produce different metrics.

**Recorded experiment:** 29 September 2026. This notebook reconstructs the verified split, retrains the eight-class baseline after the earlier checkpoint was lost, and evaluates its selected checkpoint on validation data only. See [notebook 01](01_isic2019_audit_split_baseline.ipynb) for the source-data audit.


## 1. Reconstruct the verified split

Load the two local reference CSVs and retain one copy per officially labelled image ID. Use the original seed 42 grouped split: 1,050 training, 355 validation, 356 test. Assert known-group separation.


In [ ]:
from pathlib import Path
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

root = Path("/kaggle/input")
gt_file = next(root.rglob("ISIC_2019_Training_GroundTruth.csv"))
meta_file = next(root.rglob("ISIC_2019_Training_Metadata.csv"))

gt = pd.read_csv(gt_file)
meta = pd.read_csv(meta_file)
classes = ["MEL", "NV", "BCC", "AK", "BKL", "DF", "VASC", "SCC"]

gt["image"] = gt["image"].str.upper()
meta["image"] = meta["image"].str.upper()
gt = gt.set_index("image")
meta = meta.drop_duplicates("image").set_index("image")

# Retain one copy per official image ID.
paths = {}
for p in sorted(root.rglob("*")):
    if p.suffix.lower() in {".jpg", ".jpeg", ".png"}:
        image_id = p.stem.upper()
        if image_id in gt.index and image_id not in paths:
            paths[image_id] = str(p)

rows = []
for image_id, path in sorted(paths.items()):
    positives = [c for c in classes if gt.at[image_id, c] == 1]
    if len(positives) != 1:
        continue
    lesion = meta.at[image_id, "lesion_id"] if image_id in meta.index else None
    group = (
        f"lesion:{lesion}"
        if pd.notna(lesion) and str(lesion).strip()
        else f"image:{image_id}"
    )
    rows.append((image_id, path, positives[0], lesion, group))

df = pd.DataFrame(rows, columns=["image_id", "path", "label", "lesion_id", "group"])
assert len(df) == 1761, f"Expected 1761 images, found {len(df)}"

outer = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
trainval_idx, test_idx = next(outer.split(df, df["label"], df["group"]))
trainval, test = df.iloc[trainval_idx].copy(), df.iloc[test_idx].copy()

inner = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=42)
train_idx, val_idx = next(inner.split(
    trainval, trainval["label"], trainval["group"]
))
train = trainval.iloc[train_idx].copy()
val = trainval.iloc[val_idx].copy()

train["split"], val["split"], test["split"] = "train", "val", "test"
splits = pd.concat([train, val, test], ignore_index=True)

assert not (set(train.group) & set(val.group))
assert not (set(train.group) & set(test.group))
assert not (set(val.group) & set(test.group))

output = Path("/kaggle/working/isic2019_verified_split.csv")
splits.to_csv(output, index=False)

print(pd.crosstab(splits["label"], splits["split"], margins=True))
print("Saved:", output)


split  test  train  val   All
label                        
AK       25     78   26   129
BCC      84    236   72   392
BKL      89    295   94   478
DF       24     58   22   104
MEL      30     96   28   154
NV       37     99   34   170
SCC      32    110   50   192
VASC     35     78   29   142
All     356   1050  355  1761
Saved: /kaggle/working/isic2019_verified_split.csv


## 2. Build image input pipelines

Map official classes to integers, resize RGB inputs to 128 × 128 and scale to [0,1]. Shuffle training only. The historical test pipeline is constructed but is not evaluated.


In [ ]:
import tensorflow as tf

class_names = sorted(splits["label"].unique())
class_to_num = {name: i for i, name in enumerate(class_names)}
print("Classes:", class_to_num)

def load_image(path, label):
    image = tf.io.read_file(path)
    image = tf.io.decode_image(image, channels=3, expand_animations=False)
    image.set_shape([None, None, 3])
    image = tf.image.resize(image, [128, 128])
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

def make_dataset(split_name, shuffle=False):
    part = splits[splits["split"] == split_name]
    labels = part["label"].map(class_to_num).astype("int32").to_numpy()
    ds = tf.data.Dataset.from_tensor_slices(
        (part["path"].astype(str).to_numpy(), labels)
    )
    if shuffle:
        ds = ds.shuffle(len(part), seed=42, reshuffle_each_iteration=True)
    return ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE).batch(32).prefetch(tf.data.AUTOTUNE)

train_ds = make_dataset("train", shuffle=True)
val_ds = make_dataset("val")
test_ds = make_dataset("test")

images_batch, labels_batch = next(iter(train_ds))
print("Batch shape:", images_batch.shape)
print("Labels shape:", labels_batch.shape)


Classes: {'AK': 0, 'BCC': 1, 'BKL': 2, 'DF': 3, 'MEL': 4, 'NV': 5, 'SCC': 6, 'VASC': 7}


Batch shape: (32, 128, 128, 3)
Labels shape: (32,)


## 3. Train the baseline without augmentation

Use the original CNN, seed 42, training-derived class weights, Adam 0.001, batch 32, maximum 25 epochs and early-stopping patience 5. Save the checkpoint with minimum validation loss. Epoch output below is condensed from the recorded run.


In [ ]:
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

tf.keras.utils.set_random_seed(42)

train_labels = splits.loc[splits["split"] == "train", "label"].map(class_to_num)
weights = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(len(class_names)),
    y=train_labels.to_numpy(),
)
class_weights = {i: float(w) for i, w in enumerate(weights)}
print("Class weights:", class_weights)

model = tf.keras.Sequential([
    tf.keras.Input(shape=(128, 128, 3)),

    tf.keras.layers.Conv2D(32, 3, padding="same", activation="relu"),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Conv2D(64, 3, padding="same", activation="relu"),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Conv2D(128, 3, padding="same", activation="relu"),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Conv2D(256, 3, padding="same", activation="relu"),
    tf.keras.layers.GlobalAveragePooling2D(),

    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dropout(0.35),
    tf.keras.layers.Dense(len(class_names), activation="softmax"),
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        "/kaggle/working/baseline_no_aug.keras",
        monitor="val_loss",
        save_best_only=True,
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
    ),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=25,
    class_weight=class_weights,
    callbacks=callbacks,
)

print("Best validation loss:", min(history.history["val_loss"]))
print("Epochs trained:", len(history.history["loss"]))


Class weights: {0: 1.6826923076923077, 1: 0.5561440677966102, 2: 0.4449152542372881, 3: 2.2629310344827585, 4: 1.3671875, 5: 1.3257575757575757, 6: 1.1931818181818181, 7: 1.6826923076923077}
Epoch 01: accuracy=0.0829, loss=2.0820, val_accuracy=0.0789, val_loss=2.0794
Epoch 02: accuracy=0.0867, loss=2.0793, val_accuracy=0.1099, val_loss=2.0759
Epoch 03: accuracy=0.1171, loss=2.0663, val_accuracy=0.1465, val_loss=2.0423
Epoch 04: accuracy=0.1219, loss=2.0340, val_accuracy=0.1465, val_loss=1.9596
Epoch 05: accuracy=0.1810, loss=1.8407, val_accuracy=0.1634, val_loss=1.8551
Epoch 06: accuracy=0.1629, loss=1.7221, val_accuracy=0.1831, val_loss=1.7832
Epoch 07: accuracy=0.2181, loss=1.6837, val_accuracy=0.1915, val_loss=1.7562
Epoch 08: accuracy=0.2171, loss=1.6901, val_accuracy=0.2338, val_loss=1.7447
Epoch 09: accuracy=0.2448, loss=1.7078, val_accuracy=0.2366, val_loss=1.7531
Epoch 10: accuracy=0.2305, loss=1.7415, val_accuracy=0.2704, val_loss=1.7898
Epoch 11: accuracy=0.2048, loss=1.6905,

## 4. Verify exported artifacts

Confirm that the split manifest and selected model checkpoint exist before ending the session.


In [ ]:
from pathlib import Path

files = {
    "Split CSV": Path("/kaggle/working/isic2019_verified_split.csv"),
    "Best CNN model": Path("/kaggle/working/baseline_no_aug.keras"),
}

for name, path in files.items():
    print(f"{name}: {'FOUND' if path.is_file() else 'MISSING'}")
    if path.is_file():
        print(f"  Size: {path.stat().st_size / 1024 / 1024:.2f} MB")


Split CSV: FOUND
  Size: 0.36 MB
Best CNN model: FOUND
  Size: 4.88 MB


## 5. Evaluate the selected checkpoint on validation data

Report accuracy, macro F1, balanced accuracy, per-class performance and a confusion matrix on 355 validation images. Export validation predictions and the confusion matrix. Keep the test set reserved.


In [ ]:
# Evaluate the best saved baseline on validation only (leave test untouched)
from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score, balanced_accuracy_score

best_model = tf.keras.models.load_model('/kaggle/working/baseline_no_aug.keras')
val_part = splits.loc[splits['split'] == 'val'].copy()
y_true = val_part['label'].map(class_to_num).to_numpy(dtype=int)
probabilities = best_model.predict(val_ds, verbose=0)
y_pred = probabilities.argmax(axis=1)
assert len(y_true) == len(y_pred) == 355

print('Validation accuracy:', round(accuracy_score(y_true, y_pred), 4))
print('Validation macro F1:', round(f1_score(y_true, y_pred, labels=np.arange(len(class_names)), average='macro', zero_division=0), 4))
print('Validation balanced accuracy:', round(balanced_accuracy_score(y_true, y_pred), 4))
print('\nPer-class report:')
print(classification_report(y_true, y_pred, labels=np.arange(len(class_names)), target_names=class_names, zero_division=0))
cm = pd.DataFrame(confusion_matrix(y_true, y_pred, labels=np.arange(len(class_names))), index=class_names, columns=class_names)
print('Confusion matrix (rows=true, columns=predicted):')
print(cm.to_string())
cm.to_csv('/kaggle/working/baseline_validation_confusion_matrix.csv')
val_part['predicted_label'] = [class_names[i] for i in y_pred]
val_part['confidence'] = probabilities.max(axis=1)
val_part[['image_id', 'label', 'predicted_label', 'confidence']].to_csv('/kaggle/working/baseline_validation_predictions.csv', index=False)
print('Saved validation metrics files in /kaggle/working')


Validation accuracy: 0.4141
Validation macro F1: 0.4063
Validation balanced accuracy: 0.4611

Per-class report:
              precision    recall  f1-score   support

          AK       0.20      0.62      0.30        26
         BCC       0.41      0.51      0.45        72
         BKL       0.44      0.17      0.25        94
          DF       1.00      0.05      0.09        22
         MEL       0.85      0.39      0.54        28
          NV       0.51      0.79      0.62        34
         SCC       0.35      0.26      0.30        50
        VASC       0.58      0.90      0.70        29

    accuracy                           0.41       355
   macro avg       0.54      0.46      0.41       355
weighted avg       0.49      0.41      0.39       355

Confusion matrix (rows=true, columns=predicted):
      AK  BCC  BKL  DF  MEL  NV  SCC  VASC
AK    16    4    1   0    0   0    3     2
BCC   11   37    7   0    0   0    7    10
BKL   22   33   16   0    0   8   11     4
DF    12    6   

## 6. Recorded outcome and limitations

The rerun completed 25 epochs and selected epoch 24 by minimum validation loss (1.54103). Its selected checkpoint achieved validation accuracy 41.41%, macro F1 0.4063 and balanced accuracy0.4611. These are one exploratory eight-class run, not binary results or an estimate of test performance.

The split is lesion-grouped where lesion IDs are known;324 images lack those IDs, and patient IDs are unavailable. The held-out 356-image test set remains unevaluated. Training can vary between runs and environments. Files written to `/kaggle/working` need persistent export.

See [baseline analysis](../docs/baseline.md), [archived artifacts](../artifacts/), and [current binary methodology](../docs/methodology.md).
